# Geometry Working Memory

https://www.biorxiv.org/content/10.64898/2026.08.31.748237v1

In [ ]:
# libraries

import os
import numpy as np
import pickle
import scipy


In [23]:
# functions

def compute_pca(X):

    """
    Compute PCA through eigendecomposition of covariance matrix
    pc_scores[:,0:k]: Leading k pc_scores
    eigvecs[:,0:k]:   Leading k eigenvectors

    """

    # Step 1: Center the data (subtract the mean of each feature)
    X_mean = np.mean(X, axis=0)
    X_centered = X - X_mean

    # Step 2: Compute the covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)

    # Step 3: Eigenvalue decomposition using eigh
    eigvals, eigvecs = scipy.linalg.eigh(cov_matrix)

    # Sort the eigenvectors by eigenvalues (descending order)
    sorted_indices = np.argsort(eigvals)[::-1]
    eigvals = eigvals[sorted_indices]
    eigvecs = eigvecs[:, sorted_indices]

    # Step 4: Compute principal component scores
    pc_scores = np.dot(X_centered, eigvecs)

    return eigvecs, eigvals, pc_scores



def compute_euclidean_distances(points):
    """
    Computes the pairwise Euclidean distances between all points in R^n space.

    Parameters:
    points (np.ndarray): An array of shape (num_points, n), where each row represents a point in space of n dimensions

    Returns:
    distances (np.ndarray): A matrix of pairwise Euclidean distances between all points.
    """
    num_points = points.shape[0]
    distances = np.zeros((num_points, num_points))

    for i in range(num_points):
        for j in range(i + 1, num_points):
            # Compute the Euclidean distance between points i and j
            dist = np.linalg.norm(points[i] - points[j])
            distances[i, j] = dist
            distances[j, i] = dist  # Distance is symmetric

    return distances


def extract_upper_triangle(matrix):
    """
    Extracts all the values above the diagonal (upper triangle without the diagonal elements) of a square matrix.

    Parameters:
    matrix (np.ndarray): A square matrix (n x n).

    Returns:
    np.ndarray: A 1D array containing all values above the diagonal.
    """
    # Ensure the matrix is square
    if matrix.shape[0] != matrix.shape[1]:
        raise ValueError("The input matrix must be square.")

    # Extract the upper triangle without the diagonal
    upper_triangle = matrix[np.triu_indices_from(matrix, k=1)]

    return upper_triangle


def extract_pa_vaf_pc_scores(pc_scores, k_start, k_end):

    """
    Compute principal angle and vaf between two subspaces.
    Use 4 bins of orientations.

    Inputs are PC scores after hyperalignment.

    Difference with extract_pa_vaf_aligned_pc_scores_time_resolved_orienshape(): compute for each subject, not for all subjects

    # Inputs:
        - pc_scores: matrix with PC scores with dimensions conditions by PCs (4 orientations + 3 shapes in rows)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]


    # Outputs: 
        - pa_max: vector of len(subjects) with maximum principal angles
        - pa_min: vector of len(subjects) with minimum principal angles
        - pa_avg: vector of len(subjects) with aaverage principal angles
        - vaf:    vector of len(subjects) with variance accounted for 

    # Usage:
        pa_max, pa_min, pa_avg, vaf = extract_pa_vaf(subspace1, subspace2, subjects=subjects, k_start=0, k_end=2)

    """

    pc_scores_orientation = pc_scores[0:4,:]
    pc_scores_shape = pc_scores[4:7,:]

    angle = compute_pa_best_fitting_plane(pc_scores_orientation, pc_scores_shape, k_start, k_end)
    vaf = compute_vaf_best_fitting_plane(pc_scores_orientation, pc_scores_shape, k_start, k_end)

    pa_max = np.max(angle)
    pa_min = np.min(angle)
    pa_avg = np.mean(angle)
    vaf = np.mean(vaf)

    return pa_max, pa_min, pa_avg, vaf


def compute_pa_best_fitting_plane(pc_scores1, pc_scores2, k_start, k_end):
    """

    Perform PCA on subspaces defined by pc_scores to get the best fitting plane (2 leading eigenvectors)
    and then compute principal angle between subspaces planes.

    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - pc_scores1/pc_scores2: pc_scores of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - angles: all principal angles

    """

    ### Define subspaces with k_start:k_end components
    
    pc_scores_subspace1 = pc_scores1[:, k_start:(k_end + 1)]
    pc_scores_subspace2 = pc_scores2[:, k_start:(k_end + 1)]

    ### best fitting plane
    
    if k_end - k_start > 1:

        eigvecs_subspace1, _, pc_scores_subspace1 = compute_pca(pc_scores_subspace1)
        eigvecs_subspace2, _, pc_scores_subspace2 = compute_pca(pc_scores_subspace2)

        eigvecs_subspace1 = eigvecs_subspace1[:,0:2].copy()
        eigvecs_subspace2 = eigvecs_subspace2[:,0:2].copy()
        pc_scores_subspace1 = pc_scores_subspace1[:,0:2].copy()
        pc_scores_subspace2 = pc_scores_subspace2[:,0:2].copy()

    ### Principal Angle between subspaces

    # Compute the correlation matrix C = U1.T @ U2
    C = np.dot(eigvecs_subspace1.T, eigvecs_subspace2)

    # Perform Singular Value Decomposition (SVD)
    _, singular_values, _ = np.linalg.svd(C)

    # Compute the principal angles (in radians)
    angles = np.arccos(np.clip(singular_values, -1, 1))

    # Compute angles (in degrees)
    angles = np.degrees(angles)

    return angles

def compute_vaf_best_fitting_plane(pc_scores1, pc_scores2, k_start, k_end):
    """
    
    Perform PCA on subspaces defined by pc_scores to get the best fitting plane (2 leading eigenvectors)
    and then compute VAF between subspaces planes.
    
    Inputs:
        - eigvecs1/eigvecs2: eigvecs of subspaces 1 and 2 (with all PCs in columns)
        - pc_scores1/pc_scores2: pc_scores of subspaces 1 and 2 (with all PCs in columns)
        - k_start: first component to consider (starting from 0)
        - k_end: last component to consider (starting from 0)
            e.g., k_start, k_end = 0, 2 # will subset three components [0 1 2]

    Outputs:
        - vaf: variance accounted for

    """

    ### Define subspaces with k_start:k_end components
    
    pc_scores_subspace1 = pc_scores1[:, k_start:(k_end + 1)]
    pc_scores_subspace2 = pc_scores2[:, k_start:(k_end + 1)]

    ### best fitting plane
    
    if k_end - k_start > 1:

        eigvecs_subspace1, _, pc_scores_subspace1 = compute_pca(pc_scores_subspace1)
        eigvecs_subspace2, _, pc_scores_subspace2 = compute_pca(pc_scores_subspace2)

        eigvecs_subspace1 = eigvecs_subspace1[:,0:2].copy()
        eigvecs_subspace2 = eigvecs_subspace2[:,0:2].copy()
        pc_scores_subspace1 = pc_scores_subspace1[:,0:2].copy()
        pc_scores_subspace2 = pc_scores_subspace2[:,0:2].copy()

    ### Variance Accounted For (VAF)

    ## VAF 1-to-2
    numerator_1to2 = np.linalg.norm(eigvecs_subspace2 @ eigvecs_subspace2.T @ eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    denominator_1to2 = np.linalg.norm(eigvecs_subspace1 @ pc_scores_subspace1.T, 'fro')
    vaf_1to2 = (numerator_1to2 / denominator_1to2) ** 2

    ## VAF 2-to-1
    numerator_2to1 = np.linalg.norm(eigvecs_subspace1 @ eigvecs_subspace1.T @ eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    denominator_2to1 = np.linalg.norm(eigvecs_subspace2 @ pc_scores_subspace2.T, 'fro')
    vaf_2to1 = (numerator_2to1 / denominator_2to1) ** 2

    ## Average VAF
    vaf = (vaf_1to2 + vaf_2to1) / 2

    return vaf


In [ ]:
### set paths and settings

path_root = '/path_to_local'

# settings

subjects = [f'sub_{i:02d}' for i in range(1, 50)]

X_matrices = ['x_lm']
time_windows = ['encode', 'maint', 's2']
pca_folder = 'pca_noaligned'
folder_input = 'X_matrix'
trial_type = 'correct_trials'
number_permutations = 5         # ATTENTION should be 1000
number_components_align = 10
pca_folder_surrogate = 'surrogate_pca_noaligned'
pca_folder_surrogate_aligned = 'surrogate_pca_aligned'
pca_transforms = 'pca_aligned'
k_start, k_end = 0, 2       # 0,2 means leading three PCs

outputs = [
'control_2gratings_2polygons',                                  'control_1gratings_1polygons',
'update_2gratings_relevant_2polygons_nonrelevant',              'update_1gratings_relevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_relevant',              'update_1gratings_nonrelevant_1polygons_relevant',
'inhibition_2gratings_relevant_2polygons_nonrelevant',          'inhibition_1gratings_relevant_1polygons_nonrelevant',
'inhibition_2gratings_nonrelevant_2polygons_relevant',          'inhibition_1gratings_nonrelevant_1polygons_relevant',
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      'update_1gratings_nolongerrelevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',      'update_1gratings_nonrelevant_1polygons_nolongerrelevant',
]


# Principal Component Analysis - surrogate

In [25]:
for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    # path outputs
    path_pca_outputs = os.path.join(path_root, 'results', pca_folder_surrogate, time_window + '_time_resolved')
    if not os.path.isdir(path_pca_outputs):
        os.makedirs(path_pca_outputs)

    pc_scores_dict = {}

    for sub_i in subjects:

        print(time_window + ' ' + sub_i)

        for time_segment in segments:

            # path inputs
            path_X_sub = os.path.join(path_root, 'results', folder_input, time_window + '_time_resolved', sub_i)

            # load X matrices
            filename = os.path.join(path_X_sub, 'X_dict_' + trial_type + '.pkl')
            with open(filename, 'rb') as file:
                X_dict = pickle.load(file)
                
            # loop over outputs_orientation                    
            for output in outputs:
                    
                key = time_window + '_segment' + str(time_segment) + '_' + output

                for perm_i in range(number_permutations):

                    # load X matrix
                    X = X_dict[key]

                    # shuffle rows and columns
                    X = X[np.random.permutation(X.shape[0])][:, np.random.permutation(X.shape[1])]
                                                        
                    # pca
                    eigvecs, eigvals, pc_scores = compute_pca(X)

                    # store
                    pc_scores_dict[sub_i + '_surrogate_perm' + str(perm_i) + '_pc_scores_' + key] = pc_scores[:,][:,0:number_components_align]

    # Save the dictionary to a file using pickle
    filename_dict = os.path.join(path_pca_outputs, 'pca_surrogate_' + trial_type + '.pkl')
    with open(filename_dict, 'wb') as file:
        pickle.dump(pc_scores_dict, file)


encode sub_01
encode sub_02
encode sub_03
encode sub_04
encode sub_05
encode sub_06
encode sub_07
encode sub_08
encode sub_09
encode sub_10
encode sub_11
encode sub_12
encode sub_13
encode sub_14
encode sub_15
encode sub_16
encode sub_17
encode sub_18
encode sub_19
encode sub_20
encode sub_21
encode sub_22
encode sub_23
encode sub_24
encode sub_25
encode sub_26
encode sub_27
encode sub_28
encode sub_29
encode sub_30
encode sub_31
encode sub_32
encode sub_33
encode sub_34
encode sub_35
encode sub_36
encode sub_37
encode sub_38
encode sub_39
encode sub_40
encode sub_41
encode sub_42
encode sub_43
encode sub_44
encode sub_45
encode sub_46
encode sub_47
encode sub_48
encode sub_49
maint sub_01
maint sub_02
maint sub_03
maint sub_04
maint sub_05
maint sub_06
maint sub_07
maint sub_08
maint sub_09
maint sub_10
maint sub_11
maint sub_12
maint sub_13
maint sub_14
maint sub_15
maint sub_16
maint sub_17
maint sub_18
maint sub_19
maint sub_20
maint sub_21
maint sub_22
maint sub_23
maint sub_24
ma

# Hyperalignment

Apply transforms from empirical data

In [26]:
for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    # path pca
    path_pc_scores_surrogate_noaligned = os.path.join(path_root, 'results', pca_folder_surrogate, time_window + '_time_resolved')
    path_pc_scores_surrogate_aligned = os.path.join(path_root, 'results', pca_folder_surrogate_aligned, time_window + '_time_resolved')

    if not os.path.isdir(path_pc_scores_surrogate_aligned):
        os.makedirs(path_pc_scores_surrogate_aligned)

    filename_pc_scores_surrogate_noaligned = os.path.join(path_pc_scores_surrogate_noaligned, 'pca_surrogate_' + trial_type + '.pkl')
    with open(filename_pc_scores_surrogate_noaligned, 'rb') as file:
        pc_scores_noaligned_dict = pickle.load(file)


    aligned_perm = {}

    for output in outputs:

        print('hyperalignment - ' + time_window + ' ' + output, flush=True)

        for time_segment in segments:
                                
            for perm_i in range(number_permutations):

                pc_scores_list = []

                # apply hyperalignment across subjects
                for sub_idx, sub_i in enumerate(subjects):

                    key = time_window + '_segment' + str(time_segment) + '_' + output

                    # load transforms of correct trials
                    path_transforms = os.path.join(path_root, 'results', pca_transforms, time_window + '_time_resolved', sub_i)
                    filename = os.path.join(path_transforms, 'transforms_' + trial_type + '.pkl')
                    with open(filename, 'rb') as file:
                        transforms = pickle.load(file)

                    aligned_perm[sub_i + '_surrogate_perm' + str(perm_i) + '_pc_scores_' + key] = pc_scores_noaligned_dict[sub_i + '_surrogate_perm' + str(perm_i) + '_pc_scores_' + key][:,:number_components_align] @ transforms[key]

    # store
    filename_pc_scores_aligned_dict = os.path.join(path_pc_scores_surrogate_aligned, 'pca_surrogate_' + trial_type + '.pkl')
    with open(filename_pc_scores_aligned_dict, 'wb') as file:
        pickle.dump(aligned_perm, file)



hyperalignment - encode control_2gratings_2polygons
hyperalignment - encode control_1gratings_1polygons
hyperalignment - encode update_2gratings_relevant_2polygons_nonrelevant
hyperalignment - encode update_1gratings_relevant_1polygons_nonrelevant
hyperalignment - encode update_2gratings_nonrelevant_2polygons_relevant
hyperalignment - encode update_1gratings_nonrelevant_1polygons_relevant
hyperalignment - encode inhibition_2gratings_relevant_2polygons_nonrelevant
hyperalignment - encode inhibition_1gratings_relevant_1polygons_nonrelevant
hyperalignment - encode inhibition_2gratings_nonrelevant_2polygons_relevant
hyperalignment - encode inhibition_1gratings_nonrelevant_1polygons_relevant
hyperalignment - encode update_2gratings_nolongerrelevant_2polygons_nonrelevant
hyperalignment - encode update_1gratings_nolongerrelevant_1polygons_nonrelevant
hyperalignment - encode update_2gratings_nonrelevant_2polygons_nolongerrelevant
hyperalignment - encode update_1gratings_nonrelevant_1polygons_n

# Separability - euclidean distance

In [27]:
for time_window in time_windows:

    print(time_window)
    
    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    edist_orientation = {}
    edist_shape = {}

    # loop over conditions
    for output_idx, output_i in enumerate(outputs):

        path_outputs = os.path.join(path_root, 'results', pca_folder_surrogate_aligned, time_window + '_time_resolved', 'allsubjects', trial_type)

        if not os.path.isfile(os.path.join(path_outputs, 'edist_' + time_window + '_shape_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl')):

            # empty matrix for outputs
            matrix_edist_orientation = np.empty((len(subjects), len(segments), number_permutations))
            matrix_edist_shape = np.empty((len(subjects), len(segments), number_permutations))

            # load file with pc_scores
            path_pc_scores_aligned = os.path.join(path_root, 'results', pca_folder_surrogate_aligned, time_window + '_time_resolved')
            file = os.path.join(path_pc_scores_aligned, 'pca_surrogate_' + trial_type + '.pkl')

            with open(file, 'rb') as file:
                data = pickle.load(file)

            for sub_idx, sub_i in enumerate(subjects):

                for time_idx, time_segment in enumerate(segments):

                    for perm_i in range(number_permutations):

                        key = time_window + '_segment' + str(time_segment) + '_' + output_i

                        pc_scores = data[sub_i + '_surrogate_perm' + str(perm_i) + '_pc_scores_' + key]
                        pc_scores = pc_scores[:,k_start:(k_end+1)]
                        distances_orientation = compute_euclidean_distances(pc_scores[0:4,:]) # orientation
                        matrix_edist_orientation[sub_idx, time_idx, perm_i] = np.mean(extract_upper_triangle(distances_orientation))
                        distances_shape = compute_euclidean_distances(pc_scores[4:7,:]) # shape
                        matrix_edist_shape[sub_idx, time_idx, perm_i] = np.mean(extract_upper_triangle(distances_shape))
                                    
            # assign to dict
            key_orientation = time_window + '_edistorient_' + output_i
            edist_orientation[key_orientation] = np.mean(matrix_edist_orientation, axis=2)

            key_shape = time_window + '_edistshape_' + output_i
            edist_shape[key_shape] = np.mean(matrix_edist_shape, axis=2)

    # save    
    if not os.path.isdir(path_outputs):
        os.makedirs(path_outputs)
    
    with open(os.path.join(path_outputs, 'edist_' + time_window + '_orientation_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
        pickle.dump(edist_orientation, file)

    with open(os.path.join(path_outputs, 'edist_' + time_window + '_shape_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
        pickle.dump(edist_shape, file)



encode
maint
s2


# Alignment - PA and VAF

In [28]:
for time_window in time_windows:

    print(time_window, flush=True)
    
    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    pa_dict = {}
    vaf_dict = {}

    # loop over conditions
    for output_idx, output_i in enumerate(outputs):

        path_outputs = os.path.join(path_root, 'results', pca_folder_surrogate_aligned, time_window + '_time_resolved', 'allsubjects', trial_type)
        if not os.path.isdir(path_outputs):
            os.makedirs(path_outputs)

        if not os.path.isfile(os.path.join(path_outputs, 'vaf_dict_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl')):

            # empty matrix for outputs
            matrix_pa = np.empty((len(subjects), len(segments), number_permutations))
            matrix_vaf = np.empty((len(subjects), len(segments), number_permutations))

            # load file with pc_scores
            path_pc_scores_aligned = os.path.join(path_root, 'results', pca_folder_surrogate_aligned, time_window + '_time_resolved')
            file = os.path.join(path_pc_scores_aligned, 'pca_surrogate_' + trial_type + '.pkl')

            with open(file, 'rb') as file:
                data = pickle.load(file)

            for sub_idx, sub_i in enumerate(subjects):

                for time_idx, time_segment in enumerate(segments):

                    for perm_i in range(number_permutations):

                        key = time_window + '_segment' + str(time_segment) + '_' + output_i

                        pc_scores = data[sub_i + '_surrogate_perm' + str(perm_i) + '_pc_scores_' + key]
                        pa_max, pa_min, pa_avg, vaf = extract_pa_vaf_pc_scores(pc_scores, k_start, k_end)
                        matrix_pa[sub_idx, time_idx, perm_i] = pa_max
                        matrix_vaf[sub_idx, time_idx, perm_i] = vaf


            # assign to dict
            key_pa = time_window + '_' + output_i
            pa_dict[key_pa] = np.mean(matrix_pa, axis=2)

            key_vaf = time_window + '_' + output_i
            vaf_dict[key_vaf] = np.mean(matrix_vaf, axis=2)

    # save        
    with open(os.path.join(path_outputs, 'pa_dict_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
        pickle.dump(pa_dict, file)

    with open(os.path.join(path_outputs, 'vaf_dict_subspace' + str(k_start) + 'to' + str(k_end) + '.pkl'), 'wb') as file:
        pickle.dump(vaf_dict, file)


encode
maint
s2
